# 05 — RAG Pipeline (Retrieval-Augmented Generation)

The classic "ask questions about my own documents" pattern. This notebook builds one end to end: split text → embed → store → retrieve → answer.

**Note on `langchain-community`:** you'll see many older RAG tutorials import loaders and vector stores from `langchain_community`. That package is now **being sunset** (per LangChain's own deprecation notice) in favor of standalone integration packages. This notebook avoids it — we use `InMemoryVectorStore` from `langchain_core` directly, which needs no extra package and is fully current.


In [ ]:
import os
from getpass import getpass
from langchain.chat_models import init_chat_model

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OPENAI_API_KEY: ")
MODEL_ID = "openai:gpt-4.1-mini"
model = init_chat_model(MODEL_ID, temperature=0)

## 1. Some sample documents

In a real project these would come from PDFs, Confluence pages, your Sankalp journal entries, etc. Here we hardcode a few short passages so the notebook runs standalone.


In [ ]:
from langchain_core.documents import Document

raw_docs = [
    Document(
        page_content=(
            "The Optimus Project is an internal AI adoption initiative at IDFC First Bank, "
            "focused on driving usage of the idfc-coder AI coding tool across engineering teams. "
            "It includes BA/TL/developer workflow guidelines and a squad tracker dashboard."
        ),
        metadata={"source": "optimus_project_overview"},
    ),
    Document(
        page_content=(
            "LangChain v1.0 went GA in October 2025. It introduced create_agent as the standard "
            "way to build agents, moved legacy APIs like LLMChain and AgentExecutor into a "
            "separate langchain-classic package, and split provider integrations into their own "
            "packages such as langchain-openai and langchain-anthropic."
        ),
        metadata={"source": "langchain_v1_notes"},
    ),
    Document(
        page_content=(
            "The Wealth and Demat platform at IDFC First Bank spans mutual funds, IPOs, securities, "
            "insurance, and loans, running across more than 80 microservices."
        ),
        metadata={"source": "platform_overview"},
    ),
]

## 2. Split into chunks

Even short documents benefit from consistent chunking, since it's what you'll need for real, long documents. `RecursiveCharacterTextSplitter` tries to split on paragraph/sentence boundaries before falling back to hard character cuts — it's the standard general-purpose splitter.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=40)
chunks = splitter.split_documents(raw_docs)

print(f"{len(raw_docs)} documents -> {len(chunks)} chunks")
for c in chunks:
    print("---")
    print(c.metadata["source"], ":", c.page_content[:80], "...")

## 3. Embed and store

`InMemoryVectorStore` is perfect for learning/prototyping — everything lives in process memory. For production you'd swap this for a persistent vector DB (Chroma, Pinecone, pgvector, etc.) — the rest of the pipeline code below doesn't change, since they all implement the same `VectorStore` interface.


In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = InMemoryVectorStore(embeddings)

vector_store.add_documents(chunks)
print("Indexed", len(chunks), "chunks.")

## 4. Retrieve relevant chunks for a query

`similarity_search` returns the chunks most semantically similar to your query — not keyword matching, actual meaning-based matching via embeddings.


In [ ]:
query = "What does the Optimus Project involve?"
retrieved = vector_store.similarity_search(query, k=2)

for doc in retrieved:
    print(f"[{doc.metadata['source']}] {doc.page_content}")
    print()

## 5. Wire retrieval into an LCEL chain (the full RAG chain)

This is the current, idiomatic way to build a RAG chain — no special "RetrievalQA" class needed (that class is now considered legacy). Just LCEL: retrieve context, format a prompt with it, call the model.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Answer the question using ONLY the provided context. "
     "If the context doesn't contain the answer, say you don't know.\n\nContext:\n{context}"),
    ("human", "{question}"),
])

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

retriever = vector_store.as_retriever(search_kwargs={"k": 2})

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | model
    | StrOutputParser()
)

answer = rag_chain.invoke("What does the Wealth and Demat platform at IDFC First Bank cover?")
print(answer)

## 6. Try a question the documents can't answer

A good RAG system should admit when it doesn't know, rather than hallucinating — this is why the system prompt above explicitly instructs that behavior.


In [ ]:
answer2 = rag_chain.invoke("What is the capital of France?")
print(answer2)

## 7. Turning retrieval into a tool for an agent (RAG + agent)

For more complex use cases, expose retrieval as a **tool** so an agent can decide *when* to search your documents versus answering directly or using other tools.


In [ ]:
from langchain_core.tools import tool
from langchain.agents import create_agent

@tool
def search_knowledge_base(query: str) -> str:
    """Search internal documents for information relevant to the query."""
    docs = vector_store.similarity_search(query, k=2)
    return "\n\n".join(d.page_content for d in docs)

rag_agent = create_agent(
    model=model,
    tools=[search_knowledge_base],
    system_prompt="Use the knowledge base tool when the user asks about internal projects or platforms.",
)

out = rag_agent.invoke({"messages": [{"role": "user", "content": "Tell me about the Optimus Project."}]})
print(out["messages"][-1].content)

---
### Key takeaways
- Avoid `langchain_community` loaders/vector stores where possible — that package is being sunset.
- `InMemoryVectorStore` (from `langchain_core`) needs no extra install and swaps cleanly for a real vector DB later.
- The `RetrievalQA` chain class is legacy — build RAG with plain LCEL instead (`{context, question} | prompt | model | parser`).
- Wrapping retrieval as a `@tool` lets an agent decide when to search your documents vs. do something else.

### You've completed the cookbook
You now have working, current-API examples for: chat models, prompts/chains, tools/agents, memory, structured output, and RAG. From here, the natural next step is going one level deeper into **LangGraph** directly (custom graphs, human-in-the-loop, subgraphs) once you want more control than `create_agent` gives you out of the box.
